In [1]:
import cobra
import pandas as pd
from collections import OrderedDict
from copy import deepcopy
from custom_functions_ZMRBA import*

In [2]:
from pathlib import Path

#Defining input folder
input_dir = Path("ZMRBA_inputs")
output_dir = Path("ZMRBA_outputs")

#Creating output folder
output_dir.mkdir(parents=True, exist_ok=True) 

In [3]:
# Metabolic Model Load
gsm = input_dir / 'i_ZM4_489.xml'
model = cobra.io.read_sbml_model (gsm) #sbml format of ZM GEM model

Set parameter Username
Set parameter LicenseID to value 2732945
Academic license - for non-commercial use only - expires 2026-11-04


In [4]:
# Protein Data and Amino Acid Map Data Load
protein = input_dir / 'Protein_ZM.xlsx'
df_pro = pd.read_excel (protein, sheet_name='RBA_proteins') # Read sheet anme of processed in Protein File
df_pro.index = df_pro.Gene.to_list() # make list using the gene id of proteins

AA_map = input_dir / 'Amino_Acids_Map.xlsx'
df_aamap = pd.read_excel (AA_map) # Read Amino_Acids_Map Data
df_aamap.index = df_aamap.aa_abbv.to_list() # make list using the aabreviation of amino acids

In [5]:
# Dummy Protein Data Load
dummy_protein = input_dir / 'PROTEIN_dummy_ZM.xlsx'
df_aa_dummy = pd.read_excel(dummy_protein, sheet_name = 'Dummy Protein')
df_aa_dummy.index = df_aa_dummy.aa_abbv.to_list()
dummy_medianL = int(round(df_aa_dummy.loc['A','Unnamed: 4'],0)) 
dummy_MW = round (df_aa_dummy.loc['C','Unnamed: 4'],5) + 1e-5

In [6]:
# Enzyme Data Load
enzyme = input_dir / 'Enzyme_ZM.xlsx'
df_enz = pd.read_excel (enzyme,sheet_name='Sheet1')

In [7]:
# RNA Data Load
RNA = input_dir / 'RNA_stoich_ZM.xlsx'
df_rnas = pd.read_excel (RNA, sheet_name='RNAs')
df_rnas.index = df_rnas.RNAid.to_list()

In [8]:
# Ribosome Data Load
ribosome = input_dir / 'RIBOSOME_ZM.xlsx'
df_ribo_nuc = pd.read_excel (ribosome)
#Uncomment next line if organism contains paralogs to remove them and prevent double counting 
#df_ribo_nuc = df_ribo_nuc[df_ribo_nuc.paralog.isnull()]
df_ribo_nuc.index = df_ribo_nuc.id.to_list()

In [9]:
# Biomass Data Load
biomass = input_dir / 'BIOMASS_ZM.xlsx'
df_biom = pd.read_excel (biomass, sheet_name='RBABioRxns')

In [10]:
df_eqn = pd.DataFrame(columns=['id','type','coupling_type','coupling_species','reaction'])
c = -1
# c = df_eqn.shape [0]-1, df_eqn.shape = (0,5)
#### Metabolic Network Reactions
# Exchange Reactions in Metabolic Network Reactions
for rxn in model.reactions:
    if rxn.id[:3]=='EX_':
        met = [i for i in rxn.metabolites.keys()][0]
        c +=1
        new_id = 'RXN-' + rxn.id + '_FWD-SPONT'
        df_eqn.loc[c,'id'] = new_id
        df_eqn.loc[c,'type'] = 'metabolic'
        df_eqn.loc[c,'reaction'] = 'MET-' + met.id + '-->'
        c +=1
        new_id = 'RXN-' + rxn.id + '_REV-SPONT'
        df_eqn.loc[c,'id'] = new_id
        df_eqn.loc[c,'type'] = 'metabolic'
        df_eqn.loc[c,'reaction'] = '-->'+ 'MET-' + met.id 
df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
175,RXN-EX_xyl__D_e_REV-SPONT,metabolic,NaN,NaN,-->MET-xyl__D_e
176,RXN-EX_zn2_e_FWD-SPONT,metabolic,NaN,NaN,MET-zn2_e-->
177,RXN-EX_zn2_e_REV-SPONT,metabolic,NaN,NaN,-->MET-zn2_e
178,RXN-EX_fum_e_FWD-SPONT,metabolic,NaN,NaN,MET-fum_e-->


In [11]:
# Reactions that are not exchange reactions
for i in df_enz.index:
    rxn_id = df_enz.id[i]
    _,rxn_base_id,rxn_dir,enz_id = extract_details_from_rxnid(rxn_id)
    
    if rxn_base_id[:3] == 'EX_':
        continue

    c += 1
    rxn_base = model.reactions.get_by_id(rxn_base_id)

    if rxn_base.subsystem and 'biomass' in rxn_base.subsystem.lower() and rxn_base.id != 'PEPTIDOr_ZM':
        continue

    met_dict = metabolites_dict_from_reaction_equation_RBA(rxn_base.reaction)
    met_dict = {k:v for k,v in met_dict.items() if k != ''}
    met_dict = {'MET-' + k:v for k,v in met_dict.items()}
    if rxn_dir == 'REV':
        met_dict = {k:-v for k,v in met_dict.items()}
    elif rxn_dir == 'FWD':
        None
    else:
        print("Unknown ID that indicate reaction direction, only accepting 'FWD' and 'REV'")
    
    if enz_id not in ['SPONT', 'UNKNOWN']:
        df_eqn.loc[c, 'coupling_type'] = 'rxn_enz'
        df_eqn.loc[c, 'coupling_species'] = enz_id
    
    df_eqn.loc[c, 'id'] = rxn_id
    df_eqn.loc[c, 'type'] = 'metabolic'
    df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(met_dict, arrow='-->')
df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
1278,RXN-RNFfdx_REV-ZMOenz44,metabolic,rxn_enz,ZMOenz44,MET-fdxr__4_2_c + 4 MET-h_c + MET-nad_c --> ME...
1279,RXN-NIT1b_fld_FWD-ZMOenz45,metabolic,rxn_enz,ZMOenz45,16 MET-atp_c + 8 MET-flxr_c + 16 MET-h2o_c + M...
1280,RXN-NIT1b_fld_REV-ZMOenz45,metabolic,rxn_enz,ZMOenz45,16 MET-adp_c + 8 MET-flxso_c + MET-h2_c + 6 ME...
1281,RXN-NIT1b_fdx_FWD-ZMOenz45,metabolic,rxn_enz,ZMOenz45,16 MET-atp_c + 4 MET-fdxr__4_2_c + 16 MET-h2o_...


In [12]:
### Enzyme synthesis network reaction
enz_stoich = OrderedDict()
for i in df_enz.index:
    enz_stoich[df_enz.enz[i]] = df_enz.protein_stoich[i]

c = df_eqn.shape[0] - 1
for enz_id,prot_str in enz_stoich.items():
    if prot_str == 'zeroCost':
        continue
    
    c += 1
    prot_str = prot_str.split(',')
    coeffs = OrderedDict({'PRO-' + i.split(':')[0]:-int(i.split(':')[1]) for i in prot_str})
    coeffs['ENZ-' + enz_id] = 1
    
    df_eqn.loc[c, 'id'] = 'ENZSYN-' + enz_id
    df_eqn.loc[c, 'type'] = 'enzyme'
    df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(coeffs, arrow='-->')

enzload_str = " "  

for i in df_enz.index:
    if df_enz.protein_stoich[i] == 'zeroCost':
        continue
        
    c += 1
    coeffs = OrderedDict()
    coeffs['ENZ-' + df_enz.enz[i]] = -1

    enzload_id = 'ENZLOAD-' + df_enz.id[i][4:]

    df_eqn.loc[c, 'id'] = enzload_id   
    df_eqn.loc[c, 'type'] = 'enzymeRxnLoad'
    df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(coeffs, arrow='-->')

    #get associated MW
    enz_mw = getattr(df_enz, "MW (g/mmol)")[i]

    new_line = enzload_id + "\t" + str(enz_mw) + "\n"
    enzload_str = enzload_str + new_line


enzyme_mw = output_dir / "enz_mw_g_per_mmol.txt" 
with open(enzyme_mw, 'w') as f:
    f.write(enzload_str)


df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
2573,ENZLOAD-RNFfdx_REV-ZMOenz44,enzymeRxnLoad,NaN,NaN,ENZ-ZMOenz44 -->
2574,ENZLOAD-NIT1b_fld_FWD-ZMOenz45,enzymeRxnLoad,NaN,NaN,ENZ-ZMOenz45 -->
2575,ENZLOAD-NIT1b_fld_REV-ZMOenz45,enzymeRxnLoad,NaN,NaN,ENZ-ZMOenz45 -->
2576,ENZLOAD-NIT1b_fdx_FWD-ZMOenz45,enzymeRxnLoad,NaN,NaN,ENZ-ZMOenz45 -->


In [13]:
# RIBOSOME Nucleic Synthesis Reaction
df_rnas = df_rnas[df_rnas.index.notnull()]
rnas = ['rrnas5s_c','rrnas16s_c','rrnas23s_c']
for rna in df_rnas.index:
    c += 1
    rna_stoich = OrderedDict({i:0 for i in ['MET-'+rna,'MET-atp_c','MET-ctp_c',
                                           'MET-gtp_c','MET-utp_c','MET-ppi_c']})
    rna_stoich ['RIBO-'+rna] = 1
    rna_stoich ['MET-atp_c'] = -int(df_rnas.A[rna])
    rna_stoich ['MET-ctp_c'] = -int(df_rnas.C[rna])
    rna_stoich ['MET-gtp_c'] = -int(df_rnas.G[rna])
    rna_stoich ['MET-utp_c'] = -int(df_rnas.U[rna])
    rna_stoich ['MET-ppi_c'] = int(df_rnas.loc[rna,['A','C','G','U']].sum())
    rna_stoich ['BIO-rrna'] = df_rnas.loc[rna,'MW (g/mmol)']
    
    df_eqn.loc[c, 'id'] = 'RIBOSYN-' + rna
    df_eqn.loc[c, 'type'] = 'ribosome'
    df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(rna_stoich, arrow='-->')
df_eqn

print(df_eqn)

                                  id           type coupling_type  \
0            RXN-EX_3dhq_e_FWD-SPONT      metabolic           NaN   
1            RXN-EX_3dhq_e_REV-SPONT      metabolic           NaN   
2           RXN-EX_3dhsk_e_FWD-SPONT      metabolic           NaN   
3           RXN-EX_3dhsk_e_REV-SPONT      metabolic           NaN   
4            RXN-EX_4hbz_e_FWD-SPONT      metabolic           NaN   
...                              ...            ...           ...   
2576  ENZLOAD-NIT1b_fdx_FWD-ZMOenz45  enzymeRxnLoad           NaN   
2577  ENZLOAD-NIT1b_fdx_REV-ZMOenz45  enzymeRxnLoad           NaN   
2578                RIBOSYN-rrna5s_c       ribosome           NaN   
2579               RIBOSYN-rrna16s_c       ribosome           NaN   
2580               RIBOSYN-rrna23s_c       ribosome           NaN   

     coupling_species                                           reaction  
0                 NaN                                      MET-3dhq_e-->  
1                 NaN

In [14]:
c += 1
df_ribo_nuc = df_ribo_nuc[df_ribo_nuc.index.notnull()]
ribo_stoich = OrderedDict()
rnas = ['rrna5s_c','rrna16s_c', 'rrna23s_c']
for i in df_ribo_nuc.index:
    if i in rnas:
        ribo_stoich['RIBO-' + i] = -1
    else:
        ribo_stoich['PRO-' + i] = -1
df_eqn.loc[c, 'id'] = 'RIBOSYN-ribonuc'
df_eqn.loc[c, 'type'] = 'ribosome'
df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(ribo_stoich, arrow='-->')
df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
2577,ENZLOAD-NIT1b_fdx_REV-ZMOenz45,enzymeRxnLoad,NaN,NaN,ENZ-ZMOenz45 -->
2578,RIBOSYN-rrna5s_c,ribosome,NaN,NaN,24 MET-atp_c + 33 MET-ctp_c + 33 MET-gtp_c + 2...
2579,RIBOSYN-rrna16s_c,ribosome,NaN,NaN,384 MET-atp_c + 334 MET-ctp_c + 458 MET-gtp_c ...
2580,RIBOSYN-rrna23s_c,ribosome,NaN,NaN,750 MET-atp_c + 604 MET-ctp_c + 831 MET-gtp_c ...


In [15]:
### Protein Synthesis Reactions
c = df_eqn.shape[0] - 1
for i in df_pro.index:     
    c += 1
    prot_st = OrderedDict()
    for met in [ 'MET-h2o_c',
                 'MET-pi_c', 'MET-h_c', 'MET-gtp_c',
                'MET-gdp_c']:
        prot_st[met] = 0
        
    seq = df_pro.Sequence[i][:]
    for aa in df_aamap.index:
        prot_st[df_aamap.tRNA_in[aa]] = -seq.count(aa)
        prot_st[df_aamap.tRNA_out[aa]] = seq.count(aa)
    
    prot_st['PRO-' + df_pro.Gene[i]] = 1
    
    df_eqn.loc[c, 'coupling_type'] = 'prot_ribo'
    df_eqn.loc[c, 'coupling_species'] = 'ribo'
    
    # Set protein to occupy cellular space in cytosol
    prot_st['BIO-protcyt'] = df_pro.loc[i, 'MW (g/mmol)']
        
    # Cost: Initiation: 1 GTP (IF2)
    # Elongation: 2 GTP / amino acid (EF-Tu, EF-F)
    # Termination/Recyling: 2 GTP (EF-G, RF3)
    # (Example: 601 GTP requireed for translation of 300 amino acid protein)
    for met in ['MET-gtp_c', 'MET-h2o_c']:
        prot_st[met] -= 1
    for met in ['MET-gdp_c', 'MET-pi_c', 'MET-h_c']:
        prot_st[met] += 1
                
    for met in ['MET-gtp_c', 'MET-h2o_c']:
        prot_st[met] -= 2*len(seq)
    for met in ['MET-gdp_c', 'MET-pi_c', 'MET-h_c']:
        prot_st[met] += 2*len(seq)

    df_eqn.loc[c, 'id'] = 'PROSYN-' + df_pro.Gene[i]
    df_eqn.loc[c, 'type'] = 'protein'
    
    #Output the oxidized fld/fdxcarrier to the metabolite pool
    if df_pro.Gene[i] == 'ZMO1851':
        prot_st['MET-flxso_c'] = 1
    elif df_pro.Gene[i] == 'ZMO0860':
        prot_st['MET-fdxo__4_2_c'] = 1
    elif df_pro.Gene[i] == 'ZMO2028':
        prot_st['MET-fdxo__4_2_c'] = 1
    elif df_pro.Gene[i] == 'ZMO0456':
        prot_st['MET-fdxo__4_2_c'] = 1
    elif df_pro.Gene[i] == 'ZMO1818':
        prot_st['MET-fdxo__4_2_c'] = 1
    elif df_pro.Gene[i] == 'ZMO0220':
        prot_st['MET-fdxo__4_2_c'] = 1
        
    df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(prot_st, arrow='-->')
    
    
df_eqn


,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
3113,PROSYN-ZMO0860,protein,prot_ribo,ribo,209 MET-h2o_c + 209 MET-gtp_c + 9 MET-alatrna_...
3114,PROSYN-ZMO0456,protein,prot_ribo,ribo,211 MET-h2o_c + 211 MET-gtp_c + 12 MET-alatrna...
3115,PROSYN-ZMO1818,protein,prot_ribo,ribo,129 MET-h2o_c + 129 MET-gtp_c + 7 MET-alatrna_...
3116,PROSYN-ZMO0096,protein,prot_ribo,ribo,367 MET-h2o_c + 367 MET-gtp_c + 19 MET-alatrna...


In [16]:
### Dummy protein Synthesis Reactions
prot_st = OrderedDict()
for met in [ 'MET-h2o_c',
             'MET-pi_c', 'MET-h_c', 'MET-gtp_c',
            'MET-gdp_c']:
    prot_st[met] = 0

seq = df_pro.Sequence[i][:]
for aa in df_aamap.index:
    prot_st[df_aamap.tRNA_in[aa]] = -round(df_aa_dummy.N_AA[aa], 4)
    prot_st[df_aamap.tRNA_out[aa]] = round(df_aa_dummy.N_AA[aa], 4)

for met in ['MET-gtp_c', 'MET-h2o_c']:
    prot_st[met] -= 1
for met in ['MET-gdp_c', 'MET-pi_c', 'MET-h_c']:
    prot_st[met] += 1

for met in ['MET-gtp_c', 'MET-h2o_c']:
    prot_st[met] -= 2*dummy_medianL
for met in ['MET-gdp_c', 'MET-pi_c', 'MET-h_c']:
    prot_st[met] += 2*dummy_medianL

c += 1
prot_st['BIO-protdummy'] = dummy_MW
df_eqn.loc[c, 'id'] = 'PROSYN-PROTDUMMY'
df_eqn.loc[c, 'coupling_type'] = 'prot_ribo'
df_eqn.loc[c, 'coupling_species'] = 'ribo'
df_eqn.loc[c, 'type'] = 'protein'
df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(prot_st,
                                                        arrow='-->', floatdecimal=6)
df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
3114,PROSYN-ZMO0456,protein,prot_ribo,ribo,211 MET-h2o_c + 211 MET-gtp_c + 12 MET-alatrna...
3115,PROSYN-ZMO1818,protein,prot_ribo,ribo,129 MET-h2o_c + 129 MET-gtp_c + 7 MET-alatrna_...
3116,PROSYN-ZMO0096,protein,prot_ribo,ribo,367 MET-h2o_c + 367 MET-gtp_c + 19 MET-alatrna...
3117,PROSYN-ZMO1115,protein,prot_ribo,ribo,535 MET-h2o_c + 535 MET-gtp_c + 30 MET-alatrna...


In [17]:
# Protein Waste Reactions
c = df_eqn.shape[0] - 1
for i in df_pro.index:     
    c += 1
    prot_st = OrderedDict()
    prot_st['PRO-' + df_pro.Gene[i]] = -1
    
    df_eqn.loc[c, 'id'] = 'PROWASTE-' + df_pro.Gene[i]
    df_eqn.loc[c, 'type'] = 'proteinWaste' 
    df_eqn.loc[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(prot_st, arrow='-->')
    
df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
3650,PROWASTE-ZMO0860,proteinWaste,NaN,NaN,PRO-ZMO0860 -->
3651,PROWASTE-ZMO0456,proteinWaste,NaN,NaN,PRO-ZMO0456 -->
3652,PROWASTE-ZMO1818,proteinWaste,NaN,NaN,PRO-ZMO1818 -->
3653,PROWASTE-ZMO0096,proteinWaste,NaN,NaN,PRO-ZMO0096 -->


In [18]:
#Fdx and Fld Dilution Sink Reactions
dilution_rxns = [
    {'id': 'DILUTION-fld_ox', 'type': 'dilution', 'reaction': 'MET-flxso_c -->'},
    {'id': 'DILUTION-fld_red', 'type': 'dilution', 'reaction': 'MET-flxr_c -->'},
    {'id': 'DILUTION-fdx_ox', 'type': 'dilution', 'reaction': 'MET-fdxo__4_2_c -->'},
    {'id': 'DILUTION-fdx_red', 'type': 'dilution', 'reaction': 'MET-fdxr__4_2_c -->'}
]

for rxn in dilution_rxns:
    c += 1
    df_eqn.loc[c, 'id'] = rxn['id']
    df_eqn.loc[c, 'type'] = rxn['type']
    df_eqn.loc[c, 'coupling_type'] = ''
    df_eqn.loc[c, 'coupling_species'] = ''
    df_eqn.loc[c, 'reaction'] = rxn['reaction']

df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
3654,PROWASTE-ZMO1115,proteinWaste,NaN,NaN,PRO-ZMO1115 -->
3655,DILUTION-fld_ox,dilution,,,MET-flxso_c -->
3656,DILUTION-fld_red,dilution,,,MET-flxr_c -->
3657,DILUTION-fdx_ox,dilution,,,MET-fdxo__4_2_c -->


In [19]:
### Biomass Reactions
for i in df_biom.index:
    c += 1
    df_eqn.loc[c, 'id'] = df_biom.rxn_id[i]
    df_eqn.loc[c, 'type'] = 'biomass'
    df_eqn.loc[c, 'reaction'] = df_biom.rxn_equation[i]
df_eqn

,id,type,coupling_type,coupling_species,reaction
0,RXN-EX_3dhq_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhq_e-->
1,RXN-EX_3dhq_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhq_e
2,RXN-EX_3dhsk_e_FWD-SPONT,metabolic,NaN,NaN,MET-3dhsk_e-->
3,RXN-EX_3dhsk_e_REV-SPONT,metabolic,NaN,NaN,-->MET-3dhsk_e
4,RXN-EX_4hbz_e_FWD-SPONT,metabolic,NaN,NaN,MET-4hbz_e-->
...,...,...,...,...,...
3714,BIOSYN-SMALLMOLECULE21,biomass,NaN,NaN,MET-ribflv_c --> 0.37636 BIO-ribflv_c
3715,BIOSYN-SMALLMOLECULE22,biomass,NaN,NaN,MET-so4_c --> 0.09808 BIO-so4_c
3716,BIOSYN-SMALLMOLECULE23,biomass,NaN,NaN,MET-thf_c --> 0.4434 BIO-thf_c
3717,BIOSYN-SMALLMOLECULE24,biomass,NaN,NaN,MET-thmpp_c --> 0.42229 BIO-thmpp_c


In [20]:
RBA_stoich = output_dir / 'RBA_Stoichiometry.xlsx'
df_eqn.to_excel(RBA_stoich,index=None)

In [21]:
#Getting the media conposition of the model and writing it to a text file
med_list = []

for med in model.medium:
    new_id = 'RXN-' + med + '_REV-SPONT'
    med_list.append(new_id)

med_list = ["'" + i + "'" for i in med_list]
med_list = ['/'] + med_list + ['/']
#print(med_list)

medrxn = output_dir / 'RBA_rxns_EXREV_MEDIA.txt'
with open(medrxn, 'w') as f:
    f.write('\n'.join(med_list))